In [1]:
import pandas as pd

In [2]:
df= pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/presplit/dfmodel.csv')

In [3]:
print(df.columns)

Index(['cntry', 'pplfair', 'pplhlp', 'ppltrst', 'lrscale', 'polintr', 'stfdem',
       'stfeco', 'stfgov', 'trstep', 'trstlgl', 'trstplc', 'trstplt',
       'trstprl', 'trstprt', 'trtsci_pnd', 'imbgeco', 'imwbcnt', 'happy',
       'rlgdgr', 'cntgrp_fc', 'lw_pnd', 'cnt_fc', 'rel3fc', 'happyfc',
       'confianza_promedio', 'confianza_promedio_factorizada', 'satisf_media',
       'satisf_media_factorizada', 'pintfc', 'ppl', 'ppl_fc', 'lrfc3d',
       'relbin', 'sm3d', 'sm4d', 'lr4d', 'confM4d', 'pplfair3d', 'pplhlp3d',
       'ppltrst3d', 'stfdem3d', 'stfeco3d', 'stfgov3d', 'trstep3d',
       'trstlgl3d', 'trstplc2fc', 'trstplt2fc', 'trstprl3d'],
      dtype='object')


In [4]:
colssave= ['lw_pnd','ppl','happyfc','imbgeco','cntgrp_fc','lr4d','rel3fc','pplhlp', 'pplfair','ppltrst','stfdem',
       'stfeco', 'stfgov','imwbcnt','imbgeco','trstep', 'trstlgl', 'trstplc', 'trstplt',
       'trstprl', 'trstprt', 'trtsci_pnd','polintr']
df1= df[colssave].copy()
df1.to_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/presplit/dfxgb.csv', index=False)

In [5]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62664 entries, 0 to 62663
Data columns (total 23 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   lw_pnd      62664 non-null  float64
 1   ppl         62664 non-null  float64
 2   happyfc     62664 non-null  int64  
 3   imbgeco     62664 non-null  float64
 4   cntgrp_fc   62664 non-null  int64  
 5   lr4d        62664 non-null  int64  
 6   rel3fc      62664 non-null  int64  
 7   pplhlp      62664 non-null  float64
 8   pplfair     62664 non-null  float64
 9   ppltrst     62664 non-null  float64
 10  stfdem      62664 non-null  float64
 11  stfeco      62664 non-null  float64
 12  stfgov      62664 non-null  float64
 13  imwbcnt     62664 non-null  float64
 14  imbgeco     62664 non-null  float64
 15  trstep      62664 non-null  float64
 16  trstlgl     62664 non-null  float64
 17  trstplc     62664 non-null  float64
 18  trstplt     62664 non-null  float64
 19  trstprl     62664 non-nul

In [10]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, accuracy_score
import joblib
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

def xgboost_con_hyperopt_simplificado(X, y, ruta_guardado=None, max_evals=20):
    y = y - 1
    column_names = X.columns
    X = X.apply(pd.to_numeric, errors='coerce')  
    X = X.dropna()  
    y = pd.to_numeric(y, errors='coerce')
    y = y.dropna()
    y = y[X.index]
    X = X.values  
    y = y.values  
    space = {
        'max_depth': hp.choice('max_depth', range(3, 15)),
        'learning_rate': hp.loguniform('learning_rate', -5, 0),
        'n_estimators': hp.choice('n_estimators', range(50, 300)),
        'subsample': hp.uniform('subsample', 0.5, 1),
        'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1),
        'min_child_weight': hp.choice('min_child_weight', range(1, 10)),
        'gamma': hp.uniform('gamma', 0, 1),
    }
    def objective(params):
        modelo = xgb.XGBClassifier(
            max_depth=params['max_depth'],
            learning_rate=params['learning_rate'],
            n_estimators=params['n_estimators'],
            subsample=params['subsample'],
            colsample_bytree=params['colsample_bytree'],
            min_child_weight=params['min_child_weight'],
            gamma=params['gamma'],
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss'
        )
        modelo.fit(X, y)
        y_pred = modelo.predict(X)
        score = accuracy_score(y, y_pred)
        return {'loss': -score, 'status': STATUS_OK}
    trials = Trials()
    best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=max_evals, trials=trials)
    mejor_modelo = xgb.XGBClassifier(
        max_depth=best['max_depth'],
        learning_rate=best['learning_rate'],
        n_estimators=best['n_estimators'],
        subsample=best['subsample'],
        colsample_bytree=best['colsample_bytree'],
        min_child_weight=best['min_child_weight'],
        gamma=best['gamma'],
        random_state=42,
        eval_metric='logloss'
    )
    mejor_modelo.fit(X, y)
    y_pred = mejor_modelo.predict(X)
    y_pred_proba = mejor_modelo.predict_proba(X)
    auc = roc_auc_score(y, y_pred_proba, multi_class='ovr')
    accuracy = accuracy_score(y, y_pred)

    print("\nMejor modelo encontrado (XGBoost Hyperopt):")
    print(f"best_params: {best}")
    print(f"AUC: {auc}")
    print(f"Accuracy: {accuracy}")
    if ruta_guardado:
        joblib.dump(mejor_modelo, f"{ruta_guardado}/mejor_modelo_xgboost_hyperopt.pkl")
        with open(f"{ruta_guardado}/metricas_xgboost_hyperopt.txt", "w") as f:
            f.write(f"best_params: {best}\n")
            f.write(f"AUC: {auc}\n")
            f.write(f"Accuracy: {accuracy}\n")
            f.write(f"Predictoras utilizadas: {list(column_names)}\n") #Utilizamos column_names

    return mejor_modelo
#y = df1.copy()['lr4d'] #primera variable
#X = df1.drop(['polintr', 'happyfc', 'lr4d'], axis=1).copy()
#y = df1.copy()['happyfc'] #Segunda variable
#X = df1.drop(['polintr', 'happyfc', 'lr4d'], axis=1).copy()
y = df1.copy()['polintr'] #Segunda variable
X = df1.drop(['polintr', 'happyfc', 'lr4d'], axis=1).copy()
mejor_modelo_xgboost_hyperopt = xgboost_con_hyperopt_simplificado(X, y, ruta_guardado="C:/Users/Josue/4GA.Datascience/4GA.DataScience/models/testing")


  0%|          | 0/20 [00:00<?, ?trial/s, best loss=?]

c:\Users\Josue\4GA.Datascience\.venv2\Lib\site-packages\xgboost\training.py:183: UserWarning: [04:55:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



  5%|▌         | 1/20 [00:01<00:30,  1.60s/trial, best loss: -0.4637590961317503]

c:\Users\Josue\4GA.Datascience\.venv2\Lib\site-packages\xgboost\training.py:183: UserWarning: [04:55:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



 10%|█         | 2/20 [00:02<00:19,  1.10s/trial, best loss: -0.5326343674198902]

...

 90%|█████████ | 18/20 [00:24<00:02,  1.27s/trial, best loss: -0.9737967573088216]

c:\Users\Josue\4GA.Datascience\.venv2\Lib\site-packages\xgboost\training.py:183: UserWarning: [04:55:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



 95%|█████████▌| 19/20 [00:25<00:01,  1.22s/trial, best loss: -0.9737967573088216]

c:\Users\Josue\4GA.Datascience\.venv2\Lib\site-packages\xgboost\training.py:183: UserWarning: [04:55:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



100%|██████████| 20/20 [00:27<00:00,  1.40s/trial, best loss: -0.9737967573088216]

Mejor modelo encontrado (XGBoost Hyperopt):
best_params: {'colsample_bytree': np.float64(0.9419026746837712), 'gamma': np.float64(0.391415774976725), 'learning_rate': np.float64(0.55116183608374), 'max_depth': np.int64(7), 'min_child_weight': np.int64(2), 'n_estimators': np.int64(145), 'subsample': np.float64(0.6892271315108044)}
AUC: 0.9565867770544222
Accuracy: 0.8158591854972552


In [1]:
#Todos los modelos para cada variable con la que sacaremos nuestro perfil de usuario estan entrenadas.
#Creamos un script para transferir todos los csv del proyecto a la base de datos local del repositorio sqlite3. Cerciorandonos de su funcionamiento.
import pandas as pd
import sqlite3
import os
carpetas_csv = [
    #"C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/",
    #"C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/raw/",
    "C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/presplit/"
]
ruta_db_sqlite = "C:/Users/Josue/4GA.Datascience/4GA.DataScience/src/EcoUE.db"


conexion = sqlite3.connect(ruta_db_sqlite)
cursor = conexion.cursor()


for carpeta in carpetas_csv:
    
    for archivo in os.listdir(carpeta):
        if archivo.endswith(".csv"):
            ruta_archivo = os.path.join(carpeta, archivo)
            nombre_tabla = os.path.splitext(archivo)[0]

            try:
                
                df = pd.read_csv(ruta_archivo)

                
                df.to_sql(nombre_tabla, conexion, if_exists="replace", index=False)

                print(f"Archivo {archivo} transferido a la tabla {nombre_tabla}")

            except Exception as e:
                print(f"Error al procesar el archivo {archivo}: {e}")


conexion.close()

Archivo dfmodel.csv transferido a la tabla dfmodel


In [3]:
import pandas as pd
import psycopg
import os
import configparser

config = configparser.ConfigParser()
config.read('C:/Users/Josue/4GA.Datascience/4GA.DataScience/configsql.cfg')
carpetas_csv = [
    #"C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/",
    #"C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/raw/",
    "C:/Users/Josue/4GA.Datascience/4GA.DataScience/data/processed/presplit/"
]

db_config = dict(config['postgresql'])

try:
    
    conn = psycopg.connect(
        host=db_config['host'],
        dbname=db_config['dbname'],
        user=db_config['user'],
        password=db_config['password'],
        port=db_config['port'],
        # sslmode='require',  
        connect_timeout=10
    )
    cur = conn.cursor()

    
    for carpeta in carpetas_csv:
        
        for archivo in os.listdir(carpeta):
            if archivo.endswith(".csv"):
                ruta_archivo = os.path.join(carpeta, archivo)
                nombre_tabla = os.path.splitext(archivo)[0]

                try:
                    
                    df = pd.read_csv(ruta_archivo)

                    
                    columnas = ", ".join([f'"{col}" TEXT' for col in df.columns])
                    cur.execute(f'CREATE TABLE IF NOT EXISTS "{nombre_tabla}" ({columnas});')

                    # Insertar los datos en la tabla
                    for index, row in df.iterrows():
                        valores = ", ".join([f"'{str(val)}'" for val in row.values])
                        cur.execute(f'INSERT INTO "{nombre_tabla}" VALUES ({valores});')

                    print(f"Archivo {archivo} transferido a la tabla {nombre_tabla} en PostgreSQL")

                except Exception as e:
                    print(f"Error al procesar el archivo {archivo}: {e}")

    
    conn.commit()
    cur.close()
    conn.close()

except psycopg.Error as e:
    print(f"Error connecting to PostgreSQL: {e}")

Archivo dfmodel.csv transferido a la tabla dfmodel en PostgreSQL


In [ ]:
#Ahora se realiza un pip freeze > requirements.txt y se añade junto los modelos a la carpeta de App para la app web de streamlit.
